In [17]:
import xarray as xr
import numpy as np
import pandas as pd
import glob
from datetime import datetime, timedelta
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

from aostools.climate import ComputeAnnularMode#

def calculate_djf_mean(ds):
    ds_seasons = ds.resample(time='QE-FEB').mean()
    ds_djf = ds_seasons.sel(time=(ds_seasons['time.season'] == 'DJF'))
    return ds_djf


def calculate_tercile_flags(data, var):
    # calculate terciles to cluster indices into 3 categories: -1 (negative/cold), 0 (netural), 1 (positive/warm state)
    df = data[var].to_dataframe()
    df = df.reset_index()
    df = df.set_index('time')
    tertiles = df[var].quantile([1/3, 2/3]).to_list()
    tertiles = [-float('inf')] + tertiles + [float('inf')]

    df['tercile'] = pd.cut(df[var], bins=tertiles, labels=[-1,0,1])
    return df['tercile'], tertiles

def calculate_index(data, var, time_coder=time_coder, sel_time=None):
    # calculate index and tercile flags for a given variable and path (ENSO: var='tos', NAO: var='zg')
    if sel_time is not None:
        data = data.isel(time=sel_time)
    index = (data - data.mean('time'))/ np.sqrt(data.var('time'))   
    index['tercile'], tertiles = calculate_tercile_flags(index, var)
    return index, tertiles

def anomaly_calculation(model, clim):
    ano_list = []
    for i in range(1,13):
        ano_list.append(model.sel(time=model.time[model.time.dt.month.isin(i)]) - clim.sel(month=i))
    return xr.concat(ano_list, dim='time').sortby('time')


def calculate_NAM_and_EOF(ds_ano, path):
    ### CALCULATE piControl NAM and save EOF and PC
    nam, eof, pc = ComputeAnnularMode(ds_ano.lat.values, ds_ano.plev.values, ds_ano.ua.squeeze().values, hemi='NH', choice='u', detrend='constant', eof_out=True, pc_out=True)

    nam_ds = xr.DataArray(nam, dims=['time', 'plev'], coords={'time': ds_ano.time, 'plev': ds_ano.plev})
    nam_ds.to_dataset(name='ua').to_netcdf(path + "NAM/nam.nc")

    eof = xr.DataArray(eof, dims=['plev', 'nlat'], coords={'plev': ds_ano.plev, 'nlat': ds_ano.lat[ds_ano.lat > 20].values})
    eof.to_dataset(name='eof').to_netcdf(path + "NAM/eof.nc")

    pc = xr.DataArray(pc, dims=['time', 'plev'], coords={'time': ds_ano.time, 'plev': ds_ano.plev})
    pc.to_dataset(name='pc').to_netcdf(path + "NAM/pc.nc")

    return nam, eof, pc

In [18]:
# prepare dataframe
df_all = []
outfilename = "/home/kperny/VolMIP_stratosphere/tab/nam_flags.csv"
models = ['CanESM5', 'GISS-E2-1-G', 'IPSL-CM6A-LR', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'UKESM1-0-LL']
nao_df = pd.read_csv("/home/kperny/VolMIP_stratosphere/tab/nao_flags.csv")
nam_df = nao_df[['model', 'realization', 'year']]
nam_df = nam_df.set_index(['model', 'realization'])
nam_df['nam_1000_pi'] = nam_df['nam_10_pi'] =nam_df['nam_1000_volc'] = nam_df['nam_10_volc'] = nam_df['flag_1000_pi'] = nam_df['flag_10_pi'] = nam_df['flag_1000_volc'] = nam_df['flag_10_volc'] = float(0)

In [20]:
nam_df

year  nam_1000_pi  nam_10_pi  nam_1000_volc  \
model       realization                                                      
CanESM5     r1i1p2f1     5552-02-01          0.0        0.0            0.0   
            r2i1p2f1     5557-02-01          0.0        0.0            0.0   
            r3i1p2f1     5575-02-01          0.0        0.0            0.0   
            r4i1p2f1     5622-02-01          0.0        0.0            0.0   
            r5i1p2f1     5643-02-01          0.0        0.0            0.0   
...                             ...          ...        ...            ...   
UKESM1-0-LL r21i1p1f2    3123-02-01          0.0        0.0            0.0   
            r22i1p1f2    3074-02-01          0.0        0.0            0.0   
            r23i1p1f2    3104-02-01          0.0        0.0            0.0   
            r24i1p1f2    3077-02-01          0.0        0.0            0.0   
            r25i1p1f2    3181-02-01          0.0        0.0            0.0   

                         nam_10_volc  flag_1000_pi  flag_10_pi  \
model       realization                                          
CanESM5     r1i1p2f1             0.0           0.0         0.0   
            r2i1p2f1             0.0           0.0         0.0   
            r3i1p2f1             0.0           0.0         0.0   
            r4i1p2f1             0.0           0.0         0.0   
            r5i1p2f1             0.0           0.0         0.0   
...                              ...           ...         ...   
UKESM1-0-LL r21i1p1f2            0.0           0.0         0.0   
            r22i1p1f2            0.0           0.0         0.0   
            r23i1p1f2            0.0           0.0         0.0   
            r24i1p1f2            0.0           0.0         0.0   
            r25i1p1f2            0.0           0.0         0.0   

                         flag_1000_volc  flag_10_volc  
model       realization                                
CanESM5     r1i1p2f1                0.0           0.0  
            r2i1p2f1                0.0           0.0  
            r3i1p2f1                0.0           0.0  
            r4i1p2f1                0.0           0.0  
            r5i1p2f1                0.0           0.0  
...                                 ...           ...  
UKESM1-0-LL r21i1p1f2               0.0           0.0  
            r22i1p1f2               0.0           0.0  
            r23i1p1f2               0.0           0.0  
            r24i1p1f2               0.0           0.0  
            r25i1p1f2               0.0           0.0  

[261 rows x 9 columns]

In [21]:
###########
# CanESM5 #
###########
path_pi = "/sto4/data/Intermediate/VolMIP/piControl/CanESM5/"
path_clim = path_pi + 'ua_clim_AmonZ_CanESM5_piControl_r1i1p2f1_gn_all.nc'
path_ua = path_pi + "ua_AmonZ*.nc"
path = "/sto4/data/Intermediate/VolMIP/pinatubo-full/CanESM5/ua_AmonZ*"

clim = xr.open_dataset(path_clim)
clim['time'] = clim['time.month']
clim = clim.rename({'time':'month'})

# ds = xr.open_mfdataset(path_ua)
# ds_ano = anomaly_calculation(ds, clim)
# nam, eof, pc = calculate_NAM_and_EOF(ds_ano, path_pi)
nam = xr.open_dataset(path_pi + "NAM/nam.nc")
eof = xr.open_dataset(path_pi + "NAM/eof.nc")
pc = xr.open_dataset(path_pi + "NAM/pc.nc")

nam_1000_djf = calculate_djf_mean(nam.sel(plev=100000))
nam_1000, tertiles = calculate_index(nam_1000_djf, 'ua')

nam_10_djf = calculate_djf_mean(nam.sel(plev=1000))
nam_10, tertiles = calculate_index(nam_10_djf, 'ua')

for i in range(len(glob.glob(path))):
    file = glob.glob(path + 'r' + str(i + 1) + 'i*.nc')[0]
    realization = file.split('/')[-1].split('_')[4]  
    year_post_eruption = nam_df.loc[('CanESM5', realization), 'year'][0:7]

    data_can = xr.open_dataset(file)  
    nam_df.loc[('CanESM5', realization), 'nam_1000_pi'] = nam_1000.sel(time=year_post_eruption).ua.values[0]
    nam_df.loc[('CanESM5', realization), 'nam_10_pi'] = nam_10.sel(time=year_post_eruption).ua.values[0]
    nam_df.loc[('CanESM5', realization), 'flag_1000_pi'] = nam_1000.sel(time=year_post_eruption).tercile.values[0]
    nam_df.loc[('CanESM5', realization), 'flag_10_pi'] = nam_10.sel(time=year_post_eruption).tercile.values[0]

    # calculate NAM for volc using eof and pc from clim
    data = anomaly_calculation(data_can, clim)
    nam_volc = ComputeAnnularMode(data.lat.values, data.plev.values, data.ua.squeeze().values, hemi='NH',
                                  choice='u', detrend='constant', eof_in=eof.eof.values, pc_in=pc.pc.values)
    nam_volc = xr.DataArray(nam_volc, dims=['time', 'plev'], coords={'time': data.time, 'plev': data.plev})
    data = (calculate_djf_mean(nam_volc).sel(time='1992', plev=100000) - nam_1000_djf.ua.mean('time')) /  np.sqrt(nam_1000_djf.ua.var('time'))
    nam_df.loc[('CanESM5', realization), 'nam_1000_volc'] = float(data.squeeze().values)
    nam_df.loc[('CanESM5', realization), 'flag_1000_volc'] = pd.cut([float(data.squeeze().values)], bins=tertiles, labels=[-1,0,1])[0]

    data = (calculate_djf_mean(nam_volc).sel(time='1992', plev=1000) - nam_1000_djf.ua.mean('time')) /  np.sqrt(nam_1000_djf.ua.var('time'))
    nam_df.loc[('CanESM5', realization), 'nam_10_volc'] = float(data.squeeze().values)
    nam_df.loc[('CanESM5', realization), 'flag_10_volc'] = pd.cut([float(data.squeeze().values)], bins=tertiles, labels=[-1,0,1])[0]


/tmp/ipykernel_3138955/3645944715.py:9: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  clim = xr.open_dataset(path_clim)
/tmp/ipykernel_3138955/3645944715.py:16: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  nam = xr.open_dataset(path_pi + "NAM/nam.nc")
/tmp/ipykernel_3138955/3645944715.py:18: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  pc = xr.open_datas

In [22]:
###############
# GISS-E2-1-G #
###############
path_pi = "/sto4/data/Intermediate/VolMIP/piControl/GISS-E2-1-G/"
path_clim = path_pi + 'ua_clim_AmonZ_GISS-E2-1-G_piControl_r1i1p1f2_gn_full.nc'
path_ua = path_pi + "ua_AmonZ*.nc"
path = "/sto4/data/Intermediate/VolMIP/pinatubo-full/GISS-E2-1-G/ua_AmonZ*"
model = 'GISS-E2-1-G'
clim = xr.open_dataset(path_clim)
clim['time'] = clim['time.month']
clim = clim.rename({'time':'month'})

# ds = xr.open_mfdataset(path_ua)
# ds_ano = anomaly_calculation(ds, clim)
# nam, eof, pc = calculate_NAM_and_EOF(ds_ano, path_pi)
nam = xr.open_dataset(path_pi + "NAM/nam.nc")
eof = xr.open_dataset(path_pi + "NAM/eof.nc")
pc = xr.open_dataset(path_pi + "NAM/pc.nc")

nam_1000_djf = calculate_djf_mean(nam.sel(plev=100000))
nam_1000, tertiles = calculate_index(nam_1000_djf, 'ua')

nam_10_djf = calculate_djf_mean(nam.sel(plev=1000))
nam_10, tertiles = calculate_index(nam_10_djf, 'ua')

for file in sorted(glob.glob(path)):

    realization = file.split('/')[-1].split('_')[4]  
    year_post_eruption = nam_df.loc[(model, realization), 'year'][0:7]

    data_giss = xr.open_dataset(file, decode_times=time_coder)

    nam_df.loc[(model, realization), 'nam_1000_pi'] = nam_1000.sel(time=year_post_eruption).ua.values[0]
    nam_df.loc[(model, realization), 'nam_10_pi'] = nam_10.sel(time=year_post_eruption).ua.values[0]
    nam_df.loc[(model, realization), 'flag_1000_pi'] = nam_1000.sel(time=year_post_eruption).tercile.values[0]
    nam_df.loc[(model, realization), 'flag_10_pi'] = nam_10.sel(time=year_post_eruption).tercile.values[0]      

    # calculate NAM for volc using eof and pc from clim
    data = anomaly_calculation(data_giss, clim)
    nam_volc = ComputeAnnularMode(data.lat.values, data.plev.values, data.ua.squeeze().values, hemi='NH',
                                  choice='u', detrend='constant', eof_in=eof.eof.values, pc_in=pc.pc.values)
    nam_volc = xr.DataArray(nam_volc, dims=['time', 'plev'], coords={'time': data.time, 'plev': data.plev})
    data = (calculate_djf_mean(nam_volc).sel(time='1992', plev=100000) - nam_1000_djf.ua.mean('time')) /  np.sqrt(nam_1000_djf.ua.var('time'))
    nam_df.loc[(model, realization), 'nam_1000_volc'] = float(data.squeeze().values)
    nam_df.loc[(model, realization), 'flag_1000_volc'] = pd.cut([float(data.squeeze().values)], bins=tertiles, labels=[-1,0,1])[0]

    data = (calculate_djf_mean(nam_volc).sel(time='1992', plev=1000) - nam_1000_djf.ua.mean('time')) /  np.sqrt(nam_1000_djf.ua.var('time'))
    nam_df.loc[(model, realization), 'nam_10_volc'] = float(data.squeeze().values)
    nam_df.loc[(model, realization), 'flag_10_volc'] = pd.cut([float(data.squeeze().values)], bins=tertiles, labels=[-1,0,1])[0]

/tmp/ipykernel_3138955/2569909525.py:9: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  clim = xr.open_dataset(path_clim)
/tmp/ipykernel_3138955/2569909525.py:16: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  nam = xr.open_dataset(path_pi + "NAM/nam.nc")
/tmp/ipykernel_3138955/2569909525.py:18: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  pc = xr.open_datas

In [23]:
################
# IPSL-CM6A-LR #
################
path_pi = "/sto4/data/Intermediate/VolMIP/piControl/IPSL-CM6A-LR/"
path_clim = path_pi + 'ua_clim_AmonZ_IPSL-CM6A-LR_piControl_r1i1p1f1_gr_305001-384912.nc'
path_ua = path_pi + "ua_AmonZ*.nc"
path = "/sto4/data/Intermediate/VolMIP/pinatubo-full/IPSL-CM6A-LR/ua_AmonZ*"
model = 'IPSL-CM6A-LR'

clim = xr.open_dataset(path_clim)
clim['time'] = clim['time.month']
clim = clim.rename({'time':'month'})

# ds = xr.open_mfdataset(path_ua)
# ds_ano = anomaly_calculation(ds, clim)
# nam, eof, pc = calculate_NAM_and_EOF(ds_ano, path_pi)
nam = xr.open_dataset(path_pi + "NAM/nam.nc")
eof = xr.open_dataset(path_pi + "NAM/eof.nc")
pc = xr.open_dataset(path_pi + "NAM/pc.nc")

nam_1000_djf = calculate_djf_mean(nam.sel(plev=100000))
nam_1000, tertiles = calculate_index(nam_1000_djf, 'ua')

nam_10_djf = calculate_djf_mean(nam.sel(plev=1000))
nam_10, tertiles = calculate_index(nam_10_djf, 'ua')

for i in range(len(glob.glob(path))):
    file = glob.glob(path + 'r' + str(i + 1) + 'i*.nc')[0]
    realization = file.split('/')[-1].split('_')[4]  
    year_post_eruption = nam_df.loc[(model, realization), 'year'][0:7]

    data_ipsl = xr.open_dataset(file, decode_times=time_coder)    
    
    nam_df.loc[(model, realization), 'nam_1000_pi'] = nam_1000.sel(time=year_post_eruption).ua.values[0]
    nam_df.loc[(model, realization), 'nam_10_pi'] = nam_10.sel(time=year_post_eruption).ua.values[0]
    nam_df.loc[(model, realization), 'flag_1000_pi'] = nam_1000.sel(time=year_post_eruption).tercile.values[0]
    nam_df.loc[(model, realization), 'flag_10_pi'] = nam_10.sel(time=year_post_eruption).tercile.values[0]  

    # calculate NAM for volc using eof and pc from clim
    data = anomaly_calculation(data_ipsl, clim)
    nam_volc = ComputeAnnularMode(data.lat.values, data.plev.values, data.ua.squeeze().values, hemi='NH',
                                  choice='u', detrend='constant', eof_in=eof.eof.values, pc_in=pc.pc.values)
    nam_volc = xr.DataArray(nam_volc, dims=['time', 'plev'], coords={'time': data.time, 'plev': data.plev})
    data = (calculate_djf_mean(nam_volc).sel(time='1992', plev=100000) - nam_1000_djf.ua.mean('time')) /  np.sqrt(nam_1000_djf.ua.var('time'))
    nam_df.loc[(model, realization), 'nam_1000_volc'] = float(data.squeeze().values)
    nam_df.loc[(model, realization), 'flag_1000_volc'] = pd.cut([float(data.squeeze().values)], bins=tertiles, labels=[-1,0,1])[0]

    data = (calculate_djf_mean(nam_volc).sel(time='1992', plev=1000) - nam_1000_djf.ua.mean('time')) /  np.sqrt(nam_1000_djf.ua.var('time'))
    nam_df.loc[(model, realization), 'nam_10_volc'] = float(data.squeeze().values)
    nam_df.loc[(model, realization), 'flag_10_volc'] = pd.cut([float(data.squeeze().values)], bins=tertiles, labels=[-1,0,1])[0]

/tmp/ipykernel_3138955/2191576898.py:10: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  clim = xr.open_dataset(path_clim)
/tmp/ipykernel_3138955/2191576898.py:17: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  nam = xr.open_dataset(path_pi + "NAM/nam.nc")
/tmp/ipykernel_3138955/2191576898.py:19: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  pc = xr.open_data

In [24]:
##############
# MIROC-ES2L #
##############
path_pi = "/sto4/data/Intermediate/VolMIP/piControl/MIROC-ES2L/"
path_clim = path_pi + 'ua_clim_AmonZ_MIROC-ES2L_piControl_r1i1p1f2_gn_all.nc'
path_ua = path_pi + "ua_AmonZ*.nc"
path = "/sto4/data/Intermediate/VolMIP/pinatubo-full/MIROC-ES2L/ua_AmonZ*"
model = 'MIROC-ES2L'

clim = xr.open_dataset(path_clim)
clim['time'] = clim['time.month']
clim = clim.rename({'time':'month'})

# ds = xr.open_mfdataset(path_ua)
# ds_ano = anomaly_calculation(ds, clim)
# nam, eof, pc = calculate_NAM_and_EOF(ds_ano, path_pi)
nam = xr.open_dataset(path_pi + "NAM/nam.nc")
eof = xr.open_dataset(path_pi + "NAM/eof.nc")
pc = xr.open_dataset(path_pi + "NAM/pc.nc")

nam_1000_djf = calculate_djf_mean(nam.sel(plev=100000))
nam_1000, tertiles = calculate_index(nam_1000_djf, 'ua')

nam_10_djf = calculate_djf_mean(nam.sel(plev=1000))
nam_10, tertiles = calculate_index(nam_10_djf, 'ua')

for i in range(len(glob.glob(path))):
    file = glob.glob(path + 'r' + str(i + 1) + 'i*.nc')[0]
    realization = file.split('/')[-1].split('_')[4]  
    year_post_eruption = nam_df.loc[(model, realization), 'year'][0:7]

    data_miroc = xr.open_dataset(file, decode_times=time_coder)
    
    nam_df.loc[(model, realization), 'nam_1000_pi'] = nam_1000.sel(time=year_post_eruption).ua.values[0]
    nam_df.loc[(model, realization), 'nam_10_pi'] = nam_10.sel(time=year_post_eruption).ua.values[0]
    nam_df.loc[(model, realization), 'flag_1000_pi'] = nam_1000.sel(time=year_post_eruption).tercile.values[0]
    nam_df.loc[(model, realization), 'flag_10_pi'] = nam_10.sel(time=year_post_eruption).tercile.values[0]  

    # calculate NAM for volc using eof and pc from clim
    data = anomaly_calculation(data_miroc, clim)
    nam_volc = ComputeAnnularMode(data.lat.values, data.plev.values, data.ua.squeeze().values, hemi='NH',
                                  choice='u', detrend='constant', eof_in=eof.eof.values, pc_in=pc.pc.values)
    nam_volc = xr.DataArray(nam_volc, dims=['time', 'plev'], coords={'time': data.time, 'plev': data.plev})
    data = (calculate_djf_mean(nam_volc).sel(time='1992', plev=100000) - nam_1000_djf.ua.mean('time')) /  np.sqrt(nam_1000_djf.ua.var('time'))
    nam_df.loc[(model, realization), 'nam_1000_volc'] = float(data.squeeze().values)
    nam_df.loc[(model, realization), 'flag_1000_volc'] = pd.cut([float(data.squeeze().values)], bins=tertiles, labels=[-1,0,1])[0]

    data = (calculate_djf_mean(nam_volc).sel(time='1992', plev=1000) - nam_1000_djf.ua.mean('time')) /  np.sqrt(nam_1000_djf.ua.var('time'))
    nam_df.loc[(model, realization), 'nam_10_volc'] = float(data.squeeze().values)
    nam_df.loc[(model, realization), 'flag_10_volc'] = pd.cut([float(data.squeeze().values)], bins=tertiles, labels=[-1,0,1])[0]

/tmp/ipykernel_3138955/4287470749.py:10: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  clim = xr.open_dataset(path_clim)
/tmp/ipykernel_3138955/4287470749.py:10: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  clim = xr.open_dataset(path_clim)
/tmp/ipykernel_3138955/4287470749.py:10: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  clim = xr.open_dataset(path_c

In [25]:
#################
# MPI-ESM1-2-LR #
#################
path_pi = "/sto4/data/Intermediate/VolMIP/piControl/MPI-ESM1-2-LR/"
path_clim = path_pi + 'ua_clim_AmonZ_MPI-ESM1-2-LR_piControl_r1i1p1f1_gn_all.nc'
path_ua = path_pi + "ua_AmonZ*.nc"
path = "/sto4/data/Intermediate/VolMIP/pinatubo-full/MPI-ESM1-2-LR/ua_AmonZ*"
model = 'MPI-ESM1-2-LR'

clim = xr.open_dataset(path_clim)
clim['time'] = clim['time.month']
clim = clim.rename({'time':'month'})

# ds = xr.open_mfdataset(path_ua)
# ds_ano = anomaly_calculation(ds, clim)
# nam, eof, pc = calculate_NAM_and_EOF(ds_ano, path_pi)
nam = xr.open_dataset(path_pi + "NAM/nam.nc")
eof = xr.open_dataset(path_pi + "NAM/eof.nc")
pc = xr.open_dataset(path_pi + "NAM/pc.nc")


nam_1000_djf = calculate_djf_mean(nam.sel(plev=100000))
nam_1000, tertiles = calculate_index(nam_1000_djf, 'ua')

nam_10_djf = calculate_djf_mean(nam.sel(plev=1000))
nam_10, tertiles = calculate_index(nam_10_djf, 'ua')

for i in range(len(glob.glob(path))):
    file = glob.glob(path + 'r' + str(i + 1) + 'i*.nc')[0]
    realization = file.split('/')[-1].split('_')[4]  
    year_post_eruption = nam_df.loc[(model, realization), 'year'][0:7]

    data_mpi = xr.open_dataset(file, decode_times=time_coder)
    
    nam_df.loc[(model, realization), 'nam_1000_pi'] = nam_1000.sel(time=year_post_eruption).ua.values[0]
    nam_df.loc[(model, realization), 'nam_10_pi'] = nam_10.sel(time=year_post_eruption).ua.values[0]
    nam_df.loc[(model, realization), 'flag_1000_pi'] = nam_1000.sel(time=year_post_eruption).tercile.values[0]
    nam_df.loc[(model, realization), 'flag_10_pi'] = nam_10.sel(time=year_post_eruption).tercile.values[0]  

    # calculate NAM for volc using eof and pc from clim
    data = anomaly_calculation(data_mpi, clim)
    nam_volc = ComputeAnnularMode(data.lat.values, data.plev.values, data.ua.squeeze().values, hemi='NH',
                                  choice='u', detrend='constant', eof_in=eof.eof.values, pc_in=pc.pc.values)
    nam_volc = xr.DataArray(nam_volc, dims=['time', 'plev'], coords={'time': data.time, 'plev': data.plev})
    data = (calculate_djf_mean(nam_volc).sel(time='1992', plev=100000) - nam_1000_djf.ua.mean('time')) /  np.sqrt(nam_1000_djf.ua.var('time'))
    nam_df.loc[(model, realization), 'nam_1000_volc'] = float(data.squeeze().values)
    nam_df.loc[(model, realization), 'flag_1000_volc'] = pd.cut([float(data.squeeze().values)], bins=tertiles, labels=[-1,0,1])[0]

    data = (calculate_djf_mean(nam_volc).sel(time='1992', plev=1000) - nam_1000_djf.ua.mean('time')) /  np.sqrt(nam_1000_djf.ua.var('time'))
    nam_df.loc[(model, realization), 'nam_10_volc'] = float(data.squeeze().values)
    nam_df.loc[(model, realization), 'flag_10_volc'] = pd.cut([float(data.squeeze().values)], bins=tertiles, labels=[-1,0,1])[0]



/tmp/ipykernel_3138955/2658158591.py:10: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  clim = xr.open_dataset(path_clim)
/tmp/ipykernel_3138955/2658158591.py:10: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  clim = xr.open_dataset(path_clim)
/tmp/ipykernel_3138955/2658158591.py:10: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  clim = xr.open_dataset(path_c

In [26]:
###############
# UKESM1-0-LL #
###############
path_pi = "/sto4/data/Intermediate/VolMIP/piControl/UKESM1-0-LL/"
path_clim = path_pi + 'ua_clim_AmonZ_UKESM1-0-LL_piControl_r1i1p1f2_gn_all.nc'
path_ua = path_pi + "ua_AmonZ*.nc"
path = "/sto4/data/Intermediate/VolMIP/pinatubo-full/UKESM1-0-LL/ua_AmonZ*"
model = 'UKESM1-0-LL'

clim = xr.open_dataset(path_clim)
clim['time'] = clim['time.month']
clim = clim.rename({'time':'month'})

# ds = xr.open_mfdataset(path_ua)
# ds_ano = anomaly_calculation(ds, clim)
# nam, eof, pc = calculate_NAM_and_EOF(ds_ano, path_pi)
nam = xr.open_dataset(path_pi + "NAM/nam.nc")
eof = xr.open_dataset(path_pi + "NAM/eof.nc")
pc = xr.open_dataset(path_pi + "NAM/pc.nc")


nam_1000_djf = calculate_djf_mean(nam.sel(plev=100000))
nam_1000, tertiles = calculate_index(nam_1000_djf, 'ua')

nam_10_djf = calculate_djf_mean(nam.sel(plev=1000))
nam_10, tertiles = calculate_index(nam_10_djf, 'ua')

for i in range(len(glob.glob(path))):
    file = glob.glob(path + 'r' + str(i + 1) + 'i1p1f2*')[0]
    realization = file.split('/')[-1].split('_')[4]  
    data_ukesm = xr.open_dataset(file, decode_times=time_coder)
    year_post_eruption = nam_df.loc[(model, realization), 'year'][0:7]
    
    nam_df.loc[(model, realization), 'nam_1000_pi'] = nam_1000.sel(time=year_post_eruption).ua.values[0]
    nam_df.loc[(model, realization), 'nam_10_pi'] = nam_10.sel(time=year_post_eruption).ua.values[0]
    nam_df.loc[(model, realization), 'flag_1000_pi'] = nam_1000.sel(time=year_post_eruption).tercile.values[0]
    nam_df.loc[(model, realization), 'flag_10_pi'] = nam_10.sel(time=year_post_eruption).tercile.values[0]  

    # calculate NAM for volc using eof and pc from clim
    data = anomaly_calculation(data_ukesm, clim)
    nam_volc = ComputeAnnularMode(data.lat.values, data.plev.values, data.ua.squeeze().values, hemi='NH',
                                  choice='u', detrend='constant', eof_in=eof.eof.values, pc_in=pc.pc.values)
    nam_volc = xr.DataArray(nam_volc, dims=['time', 'plev'], coords={'time': data.time, 'plev': data.plev})
    data = (calculate_djf_mean(nam_volc).sel(time='1992', plev=100000) - nam_1000_djf.ua.mean('time')) /  np.sqrt(nam_1000_djf.ua.var('time'))
    nam_df.loc[(model, realization), 'nam_1000_volc'] = float(data.squeeze().values)
    nam_df.loc[(model, realization), 'flag_1000_volc'] = pd.cut([float(data.squeeze().values)], bins=tertiles, labels=[-1,0,1])[0]

    data = (calculate_djf_mean(nam_volc).sel(time='1992', plev=1000) - nam_1000_djf.ua.mean('time')) /  np.sqrt(nam_1000_djf.ua.var('time'))
    nam_df.loc[(model, realization), 'nam_10_volc'] = float(data.squeeze().values)
    nam_df.loc[(model, realization), 'flag_10_volc'] = pd.cut([float(data.squeeze().values)], bins=tertiles, labels=[-1,0,1])[0]

/tmp/ipykernel_3138955/2282702678.py:10: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  clim = xr.open_dataset(path_clim)
/tmp/ipykernel_3138955/2282702678.py:17: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  nam = xr.open_dataset(path_pi + "NAM/nam.nc")
/tmp/ipykernel_3138955/2282702678.py:19: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  pc = xr.open_data

In [ ]:
# save as csv
nam_df.reset_index().to_csv(outfilename, index=False)